In [1]:
import pandas as pd 
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [2]:
nltk.download("punkt_tab")
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\yugyg\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\yugyg\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
df = pd.read_csv("b.csv")

In [4]:
df.columns
df.head()
df["category"].unique()

<StringArray>
['tech', 'business', 'sport', 'entertainment', 'politics']
Length: 5, dtype: str

In [5]:
stop_words = set(stopwords.words('english'))

In [6]:
import string
from nltk.tokenize import word_tokenize

def preprocessing(text):
    text = str(text).lower()   
    text = text.replace('-', ' ').replace('/', ' ')
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)  
    tokens = [word for word in tokens if word.isalpha() and word not in stop_words]  
    
    return tokens



In [ ]:
def BoW(text,vocab):
    biglist = []
    for item in text:
        lst = np.zeros(len(vocab))

        for i, word in enumerate(vocab):
            lst[i] = item.count(word)

        biglist.append(lst)    
    return np.array(biglist)    

In [18]:
def calculate_tf(document, vocab):
    tf_matrix = np.zeros((len(document), len(vocab)))
    
    for doc_idx, doc in enumerate(document):
        length = len(doc)
        if length == 0:
            continue 
            
        for i, word in enumerate(vocab):
            if word in doc:
                n = doc.count(word)
                tf_matrix[doc_idx, i] = n / length
                
    return tf_matrix


In [19]:
def calculate_idf(corpus, vocab):
    N = len(corpus)
    idf = np.zeros(len(vocab))

    for i, word in enumerate(vocab):
        df = sum(1 for document in corpus if word in document)
        idf[i] = np.log((N + 1) / (df + 1)) + 1

    return idf

In [20]:
def calculate_tfidf(document, vocab,idf):
    tf = calculate_tf(document, vocab)
    tfidf = tf * idf
    return tfidf

In [21]:
X = df["text"]
y = df["category"]

In [ ]:
cleaned_text = X.apply(preprocessing)

X_train_text_raw, X_test_text_raw, y_train, y_test = train_test_split(
    cleaned_text, y, test_size=0.2, random_state=42, stratify=y
)
X_train_text = list(X_train_text_raw)
X_test_text = list(X_test_text_raw)

vocab = sorted(list(set([word for doc in X_train_text for word in doc])))

tf_train = calculate_tf(X_train_text, vocab)
idf_train = calculate_idf(X_train_text, vocab)
X_train_tfidf = tf_train * idf_train

tf_test = calculate_tf(X_test_text, vocab)
X_test_tfidf = tf_test * idf_train

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)
print(classification_report(y_test, y_pred))


               precision    recall  f1-score   support

     business       0.95      0.98      0.97       102
entertainment       1.00      0.97      0.99        77
     politics       0.96      0.98      0.97        84
        sport       0.98      1.00      0.99       102
         tech       0.99      0.94      0.96        80

     accuracy                           0.98       445
    macro avg       0.98      0.97      0.98       445
 weighted avg       0.98      0.98      0.98       445



In [ ]:
def build_vocab(text):
    local_vocab = set()  
    
    for lst in text:
        for token in lst:
            local_vocab.add(token) 
            
    return local_vocab        


In [26]:
cleaned_text = X.apply(preprocessing)

voc = sorted(build_vocab(cleaned_text))
biglist = BoW(cleaned_text, voc)

print(biglist)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [27]:
voc = sorted(voc)

idf = calculate_idf(cleaned_text, voc)

tfidf_matrix = calculate_tfidf(
    cleaned_text, voc, idf
)

print(tfidf_matrix.shape)

(2225, 27779)


In [28]:
def top_10_words(category, y, tfidf_matrix, voc):
    mask = (y == category).to_numpy()

    category_tfidf = tfidf_matrix[mask]

    avg_tfidf = category_tfidf.mean(axis=0)

    top_indices = np.argsort(avg_tfidf)[::-1][:10]

    result = pd.DataFrame({
        "category": category,
        'word': [voc[i] for i in top_indices],
        'average_tfidf': avg_tfidf[top_indices]
        
    })

    return result

In [29]:
print(y.unique())
top_10_words(y.unique()[0], y, tfidf_matrix, voc)

<StringArray>
['tech', 'business', 'sport', 'entertainment', 'politics']
Length: 5, dtype: str


,category,word,average_tfidf
0,tech,people,0.017232
1,tech,said,0.016751
2,tech,mobile,0.015607
3,tech,users,0.014638
4,tech,software,0.014396
5,tech,technology,0.013078
6,tech,microsoft,0.011826
7,tech,net,0.011661
8,tech,computer,0.011529
9,tech,digital,0.011190


In [30]:
tf_matrix = calculate_tf(cleaned_text, voc)

avg_tf = tf_matrix.mean(axis=0)

In [31]:
word_stats = pd.DataFrame({
    'word': voc,
    'average_tf': avg_tf,
    'idf': idf
})

In [32]:
high_tf_low_idf = word_stats[
    (word_stats['average_tf'] > word_stats['average_tf'].median()) &
    (word_stats['idf'] < word_stats['idf'].median())
].sort_values('average_tf', ascending=False)

low_tf_high_idf = word_stats[
    (word_stats['average_tf'] < word_stats['average_tf'].median()) &
    (word_stats['idf'] > word_stats['idf'].median())
].sort_values('idf', ascending=False)

In [33]:
print("High TF, low IDF:")
print(high_tf_low_idf.head(10))

print("\nLow TF, high IDF:")
print(low_tf_high_idf.head(10))

High TF, low IDF:
         word  average_tf       idf
21452    said    0.015595  1.164159
27616    year    0.005554  1.625413
16286      mr    0.005551  2.034664
27473   would    0.005033  1.663056
763      also    0.004586  1.564344
26293      us    0.004377  1.969809
16709     new    0.003993  1.818370
17259     one    0.003786  1.732548
18150  people    0.003531  2.024601
14015    last    0.003283  1.887945

Low TF, high IDF:
         word    average_tf       idf
0          aa  1.366074e-06  8.014814
4        aadc  4.764363e-06  8.014814
6      aaltra  3.036745e-06  8.014814
7       aamir  1.417786e-06  8.014814
8         aan  2.047554e-07  8.014814
9        aara  9.791682e-07  8.014814
10     aarhus  1.244981e-06  8.014814
17      abate  3.808798e-06  8.014814
18  abatement  1.395771e-06  8.014814
24      abbot  1.888396e-06  8.014814


In [ ]:
def BoW(text, vocab):
    rows = []
    for item in text:
        lst = np.zeros(len(vocab))
        for i, word in enumerate(vocab):
            lst[i] = item.count(word)
        rows.append(lst)
    return np.array(rows)



In [36]:

X_train_bow = BoW(X_train_text, vocab)
X_test_bow = BoW(X_test_text, vocab)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_bow, y_train)

y_pred = model.predict(X_test_bow)
print(classification_report(y_test, y_pred))

               precision    recall  f1-score   support

     business       0.99      0.97      0.98       102
entertainment       0.95      0.99      0.97        77
     politics       0.98      0.99      0.98        84
        sport       0.99      1.00      1.00       102
         tech       0.99      0.95      0.97        80

     accuracy                           0.98       445
    macro avg       0.98      0.98      0.98       445
 weighted avg       0.98      0.98      0.98       445

